# Entropic OT baseline for simulation experiments

This notebook runs a Pooladian--Niles-Weed style entropic OT baseline on the simulation scenarios used in this project. The reusable implementation lives in `simulation_code/entropic_ot_baseline.py`; this notebook only configures and calls that code.

For each empirical source/target pair, the code computes an entropy-regularized OT coupling with POT/Sinkhorn. Training-sample predictions use barycentric projection. Test-sample predictions use the learned Sinkhorn target scaling `v` in normalized entropic weights against the target samples. If POT does not return `v`, the code falls back to normalized Gibbs kernel weights and records that limitation in metadata.


In [ ]:
from pathlib import Path
import csv
import subprocess
import sys
from collections import defaultdict

import numpy as np

ROOT = Path.cwd()
SCRIPT = ROOT / "simulation_code" / "entropic_ot_baseline.py"
OUTPUT_ROOT = ROOT / "simulation_results" / "entropic_ot"


## Pilot command

The local pilot is intentionally small and capped at nine workers. Seeds follow the neural simulation convention: `model_i` uses `20260527 + i`; evaluation data uses seed `20260527`.


In [ ]:
cmd = [
    sys.executable,
    str(SCRIPT),
    "--dimensions", "10",
    "--sample-sizes", "100",
    "--measures", "normal", "t",
    "--transforms", "CDF", "piecewise_linear", "quadratic",
    "--model-start", "0",
    "--model-end", "20",
    "--n-jobs", "9",
    "--epsilon-rule", "pnw",
    "--epsilon-c", "1.0",
    "--alpha", "1.0",
    "--test-size", "10000",
    "--evaluation-seed", "20260527",
    "--sinkhorn-method", "sinkhorn_log",
]
print(" ".join(cmd))
# subprocess.run(cmd, check=True)


## Result summary

After running the pilot, this cell reads the per-model L2 CSV files and prints scenario-level summaries without introducing extra dependencies beyond numpy.


In [ ]:
rows = []
for csv_path in sorted(OUTPUT_ROOT.glob("d=*/**/L2_error_measure_P=*_transform_method=*.csv")):
    scenario = csv_path.parent
    with csv_path.open(newline="") as f:
        for row in csv.DictReader(f):
            row = dict(row)
            row["scenario"] = str(scenario.relative_to(OUTPUT_ROOT))
            row["L2_loss"] = float(row["L2_loss"])
            row["epsilon"] = float(row["epsilon"])
            rows.append(row)

if rows:
    by_scenario = defaultdict(list)
    for row in rows:
        by_scenario[row["scenario"]].append(row)
    for scenario, scenario_rows in sorted(by_scenario.items()):
        losses = np.array([row["L2_loss"] for row in scenario_rows], dtype=float)
        print(
            scenario,
            "count=", losses.size,
            "mean=", float(losses.mean()),
            "sd=", float(losses.std(ddof=1)) if losses.size > 1 else float("nan"),
            "median=", float(np.median(losses)),
            "epsilon=", scenario_rows[0]["epsilon"],
            "formula=", scenario_rows[0]["evaluation_formula"],
        )
else:
    print("No entropic OT result CSVs found yet.")
